# Project 9: Industrial Equipment Failure Forecasting
IDEAS - Institute of Data Engineering, Analytics and Science Foundation | Autumn Internship Program 2026 | Mentor: Dr. Chandan Biswas

## Question 1: Data Loading & Initial Inspection

In [1]:
import pandas as pd
import numpy as np

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
try:
    df = pd.read_csv(URL)
except Exception as e:
    # Fallback: modern ucimlrepo package (id 601 = AI4I 2020)
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=601)
    df = pd.concat([ds.data.features, ds.data.targets], axis=1)
    if "UDI" not in df.columns:
        df.insert(0, "UDI", range(1, len(df) + 1))
    if "Product ID" not in df.columns:
        df.insert(1, "Product ID", "")

df = df.drop(columns=["UDI", "Product ID"])
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Question 2: Missing Values & One-Hot Encoding

In [2]:
print(df.isnull().sum())

type_dummies = pd.get_dummies(df["Type"], prefix="Type", dtype=int)
df = pd.concat([df.drop(columns=["Type"]), type_dummies], axis=1)
df.head()

Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_H,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,0,0,1
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,0,1,0
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,0,1,0
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,0,1,0
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,0,1,0


## Question 3: Target Distribution & Analytical Reasoning

In [3]:
dist = df["Machine failure"].value_counts(normalize=True) * 100
print(f"Normal operation (0): {dist[0]:.2f}%")
print(f"Equipment failure (1): {dist[1]:.2f}%")

print("""
Analysis: Only about 3.4% of observations are failures, so the classes are severely imbalanced.
Accuracy = (TP + TN) / (TP + TN + FP + FN) weights every sample equally, so it is dominated by
the majority class. A trivial model that always predicts "no failure" scores roughly 96.6%
accuracy while detecting zero failures (recall = 0 for the positive class). High accuracy
therefore says nothing about the ability to find the rare, costly events we actually care about.
Metrics such as recall, precision, F1, PR-AUC and ROC-AUC, or a cost-based metric, are more
appropriate because they evaluate performance on the minority class explicitly.
""")

Normal operation (0): 96.61%
Equipment failure (1): 3.39%

Analysis: Only about 3.4% of observations are failures, so the classes are severely imbalanced.
Accuracy = (TP + TN) / (TP + TN + FP + FN) weights every sample equally, so it is dominated by
the majority class. A trivial model that always predicts "no failure" scores roughly 96.6%
accuracy while detecting zero failures (recall = 0 for the positive class). High accuracy
therefore says nothing about the ability to find the rare, costly events we actually care about.
Metrics such as recall, precision, F1, PR-AUC and ROC-AUC, or a cost-based metric, are more
appropriate because they evaluate performance on the minority class explicitly.



## Question 4: Feature Engineering

In [4]:
df["Temperature_Difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["Power_Estimate"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]
df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_H,Type_L,Type_M,Temperature_Difference,Power_Estimate
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,0,0,1,10.5,66382.8
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,0,1,0,10.5,65190.4
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,0,1,0,10.4,74001.2
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,0,1,0,10.4,56603.5
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,0,1,0,10.5,56320.0


## Question 5: Outlier Detection (IQR) on Rotational speed

In [5]:
col = df["Rotational speed [rpm]"]
Q1 = col.quantile(0.25)
Q3 = col.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Lower bound={lower:.2f}, Upper bound={upper:.2f}")

outlier_count = int(((col < lower) | (col > upper)).sum())
print(outlier_count)

Q1=1423.0, Q3=1612.0, IQR=189.0
Lower bound=1139.50, Upper bound=1895.50
418


## Question 6: Correlation Analysis & Feature Selection

In [6]:
corr = df.select_dtypes(include=[np.number]).corr()
target_corr = corr["Machine failure"].abs()
exclude = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
top3 = target_corr.drop(labels=exclude).sort_values(ascending=False).head(3)
print(top3)

Torque [Nm]               0.191321
Power_Estimate            0.176039
Temperature_Difference    0.111676
Name: Machine failure, dtype: float64


## Question 7: Data Splitting and Standardization

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

drop_cols = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
X = df.drop(columns=drop_cols)
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
X_train_scaled.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_H,Type_L,Type_M,Temperature_Difference,Power_Estimate
9254,-0.854066,-0.609589,0.427634,-0.892696,1.375035,-0.333796,0.821609,-0.658943,0.806356,-0.954812
1561,-0.904014,-1.080528,-0.834945,1.382187,0.457620,-0.333796,0.821609,-0.658943,0.204476,1.442085
1670,-0.904014,-1.484190,-0.059677,-0.892696,1.359218,-0.333796,0.821609,-0.658943,-0.397404,-1.223472
6087,0.444571,0.534121,0.333495,-0.702288,-1.598655,-0.333796,-1.217123,1.517582,-0.096464,-0.708474
6669,0.694309,0.332290,0.178441,-0.612094,1.580663,-0.333796,0.821609,-0.658943,-0.898970,-0.660383


## Question 8: Baseline Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
log_reg

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

## Question 9: Cross-Validation and Baseline Evaluation

In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

cv_scores = cross_val_score(log_reg, X_train_scaled, y_train, cv=5, scoring="roc_auc")
print("Mean 5-fold CV ROC-AUC:", cv_scores.mean())

y_pred_lr = log_reg.predict(X_test_scaled)
print(classification_report(y_test, y_pred_lr))

Mean 5-fold CV ROC-AUC: 0.9239508781101857
              precision    recall  f1-score   support

           0       0.99      0.86      0.92      1939
           1       0.15      0.79      0.26        61

    accuracy                           0.86      2000
   macro avg       0.57      0.83      0.59      2000
weighted avg       0.97      0.86      0.90      2000



## Question 10: Hyperparameter Tuning via GridSearchCV

In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [0.01, 0.1, 1, 10]}
grid = GridSearchCV(
    LogisticRegression(class_weight="balanced", max_iter=1000),
    param_grid, cv=5, scoring="roc_auc"
)
grid.fit(X_train_scaled, y_train)
print("Best parameters:", grid.best_params_)
print("Best estimator:", grid.best_estimator_)

Best parameters: {'C': 10}
Best estimator: LogisticRegression(C=10, class_weight='balanced', max_iter=1000)


## Question 11: MLP Initialization

In [11]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                    max_iter=500, random_state=42)
mlp

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",500
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


## Question 12: MLP Training and ROC-AUC

In [12]:
from sklearn.metrics import roc_auc_score

mlp.fit(X_train_scaled, y_train)
mlp_probs = mlp.predict_proba(X_test_scaled)[:, 1]
print("MLP ROC-AUC:", roc_auc_score(y_test, mlp_probs))

MLP ROC-AUC: 0.9711698610911489


## Question 13: Custom Business Metric

In [13]:
from sklearn.metrics import confusion_matrix

def calculate_financial_penalty(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return fn * 10000 + fp * 500

y_pred_mlp = mlp.predict(X_test_scaled)
print(f"Total financial penalty (default MLP predictions): ${calculate_financial_penalty(y_test, y_pred_mlp):,}")

Total financial penalty (default MLP predictions): $235,000


## Question 14: Decision Threshold Optimization

In [14]:
min_cost = float("inf")
best_threshold = None

for t in np.round(np.arange(0.1, 1.0, 0.1), 1):
    preds = (mlp_probs >= t).astype(int)
    cost = calculate_financial_penalty(y_test, preds)
    if cost < min_cost:
        min_cost = cost
        best_threshold = t

print("Optimal threshold:", best_threshold)
print(f"Minimum cost: ${min_cost:,}")

Optimal threshold: 0.1
Minimum cost: $170,500


## Question 15: Permutation Importance & Analytical Reasoning

In [15]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(mlp, X_test_scaled, y_test, n_repeats=5,
                              scoring="roc_auc", random_state=42)
imp = pd.Series(perm.importances_mean, index=X_test_scaled.columns).sort_values(ascending=False)
print("Top 3 most influential features:")
print(imp.head(3))

print("""
Analysis: The Pearson correlation in Question 6 measures only the bivariate, linear association
between one feature and the target, in isolation. It cannot capture non-linear relationships,
feature interactions, or redundancy among correlated predictors. Permutation importance is
model-based: it shuffles one feature at a time on held-out data and measures the drop in the
model's ROC-AUC, so it reflects how much the trained MLP actually relies on that feature,
including through non-linear effects and interactions (e.g. torque combined with rotational
speed). It is computed on unseen test data, is tied to the metric we care about, and is
model-agnostic. Features with weak linear correlation can therefore rank highly, and the
repeats (n_repeats=5) give a more stable estimate than a single correlation coefficient.
""")

Top 3 most influential features:
Torque [Nm]               0.406784
Temperature_Difference    0.134037
Tool wear [min]           0.133848
dtype: float64

Analysis: The Pearson correlation in Question 6 measures only the bivariate, linear association
between one feature and the target, in isolation. It cannot capture non-linear relationships,
feature interactions, or redundancy among correlated predictors. Permutation importance is
model-based: it shuffles one feature at a time on held-out data and measures the drop in the
model's ROC-AUC, so it reflects how much the trained MLP actually relies on that feature,
including through non-linear effects and interactions (e.g. torque combined with rotational
speed). It is computed on unseen test data, is tied to the metric we care about, and is
model-agnostic. Features with weak linear correlation can therefore rank highly, and the
repeats (n_repeats=5) give a more stable estimate than a single correlation coefficient.

